In [2]:
# ============================================================
# Project: EuroTrans Analytics
# Notebook: 04_Gold_Dimensions
# Layer: Gold
#
# Description:
# Create Gold Dimension tables from the Silver layer.
# Enrich Route Dimension with warehouse information.
# ============================================================

# ------------------------------------------------------------
# Import Libraries
# ------------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, col

# ------------------------------------------------------------
# Create Spark Session
# ------------------------------------------------------------

spark = SparkSession.builder.getOrCreate()

print("=" * 60)
print("Starting Gold Dimension creation...")
print("=" * 60)

# ============================================================
# CUSTOMER
# ============================================================

customer = spark.table("silver_customer")

(
    customer.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_customer")
)

print("✅ gold_dim_customer created")

# ============================================================
# PRODUCT
# ============================================================

product = spark.table("silver_product")

(
    product.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_product")
)

print("✅ gold_dim_product created")

# ============================================================
# CARRIER
# ============================================================

carrier = spark.table("silver_carrier")

(
    carrier.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_carrier")
)

print("✅ gold_dim_carrier created")

# ============================================================
# DATE
# ============================================================

date = spark.table("silver_date")

(
    date.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_date")
)

print("✅ gold_dim_date created")

# ============================================================
# WAREHOUSE
# ============================================================

warehouse = spark.table("silver_warehouse")

(
    warehouse.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_warehouse")
)

print("✅ gold_dim_warehouse created")

# ============================================================
# ROUTE (Enriched)
# ============================================================

route = spark.table("silver_route")

warehouse_origin = (
    warehouse
    .select(
        col("WarehouseID").alias("OriginWarehouseID"),
        col("WarehouseName").alias("OriginWarehouse"),
        col("City").alias("OriginCity"),
        col("Country").alias("OriginCountry")
    )
)

warehouse_destination = (
    warehouse
    .select(
        col("WarehouseID").alias("DestinationWarehouseID"),
        col("WarehouseName").alias("DestinationWarehouse"),
        col("City").alias("DestinationCity"),
        col("Country").alias("DestinationCountry")
    )
)

gold_route = (
    route

    .join(
        warehouse_origin,
        "OriginWarehouseID",
        "left"
    )

    .join(
        warehouse_destination,
        "DestinationWarehouseID",
        "left"
    )

    .withColumn(
        "RouteName",
        concat_ws(
            " → ",
            col("OriginCity"),
            col("DestinationCity")
        )
    )
)

(
    gold_route.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_route")
)

print("✅ gold_dim_route created")
print("=" * 60)

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

cost_data = [
    ("Profit", 1),
    ("Fuel", 2),
    ("Driver", 3),
    ("Toll", 4),
    ("Maintenance", 5),
    ("Handling", 6)
]

schema = StructType([
    StructField("CostType", StringType(), False),
    StructField("SortOrder", IntegerType(), False)
])

gold_dim_cost = spark.createDataFrame(cost_data, schema)

(
    gold_dim_cost.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_cost")
)

print("gold_dim_cost created successfully")
display(gold_dim_cost)
print("Gold Dimension Layer successfully created.")
print("=" * 60)

StatementMeta(, 00495c9c-0a76-42b9-b813-7289934bb07f, 4, Finished, Available, Finished, False)

Starting Gold Dimension creation...
✅ gold_dim_customer created
✅ gold_dim_product created
✅ gold_dim_carrier created
✅ gold_dim_date created
✅ gold_dim_warehouse created
✅ gold_dim_route created
gold_dim_cost created successfully


SynapseWidget(Synapse.DataFrame, 0b315f40-abdf-4362-9719-c82713c4e861)

Gold Dimension Layer successfully created.
